In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import warnings
warnings.filterwarnings("ignore")

train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

# q1
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
print(f"Q1: {label_map[train_df.loc[150, 'answer']]}")

# q2
prompt_0 = str(train_df.loc[0, 'prompt'])
option_B_0 = str(train_df.loc[0, 'B'])
formatted_string = prompt_0 + " [SEP] " + option_B_0
print(f"Q2: {len(formatted_string)}")

In [ ]:
# q7
model_name = "bert-base-uncased"

base_model = AutoModelForMultipleChoice.from_pretrained(model_name)

# LoRA config
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
lora_model = get_peft_model(base_model, peft_config)

trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print(f"Q7: {trainable_params}")

In [ ]:
# q10
tiny_df = train_df.head(32).copy()
tokenizer = AutoTokenizer.from_pretrained(model_name)
option_letters = ['A', 'B', 'C', 'D', 'E']

# hf dataset
def preprocess(examples):
    first_sentences = [[prompt] * 5 for prompt in examples['prompt']]
    second_sentences = [[str(examples[opt][i]) for opt in option_letters] for i in range(len(examples['prompt']))]
    
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])
    
    tokenized = tokenizer(first_sentences, second_sentences, truncation=True, max_length=64, padding="max_length")
    return {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}

tiny_dataset = Dataset.from_pandas(tiny_df)
tiny_dataset = tiny_dataset.map(preprocess, batched=True)
tiny_dataset = tiny_dataset.map(lambda x: {'labels': label_map[x['answer']]})

# trainer
training_args = TrainingArguments(
    output_dir="./tiny_test",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    report_to="none",
    logging_steps=1
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tiny_dataset
)

trainer.train()

row_0_input = tiny_dataset[0]
input_ids = torch.tensor([row_0_input['input_ids']]).to(lora_model.device)
attention_mask = torch.tensor([row_0_input['attention_mask']]).to(lora_model.device)
token_type_ids = torch.tensor([row_0_input['token_type_ids']]).to(lora_model.device)

lora_model.eval()
with torch.no_grad():
    outputs = lora_model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
    logits = outputs.logits
    
    probabilities = torch.nn.functional.softmax(logits, dim=-1)

prob_E = probabilities[0][4].item()
print(f"Q10: {prob_E:.4f}")